# Common setup

Run these cells first.

In [ ]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
from collections import Counter

RUNPOD_ROOT = Path("/workspace/SKN27-FINAL-3Team")
PROJECT_ROOT = None
if RUNPOD_ROOT.exists() and (RUNPOD_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = RUNPOD_ROOT
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "ai").exists() and (candidate / "storage").exists():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("project root not found")
MANIFEST_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/manifests"
RAW_VIDEO_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/raw_videos"
CLIP_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/clips_5s"
MODEL_DIR = PROJECT_ROOT / "storage/vision/models"
SAMPLE_MANIFEST = MANIFEST_DIR / "sample_700_coarse_manifest.csv"
DOWNLOAD_MANIFEST = MANIFEST_DIR / "train_700_download_manifest.csv"
CLIP_MANIFEST = MANIFEST_DIR / "train_700_clip_manifest_5s.csv"

PER_LABEL = 700
SEED = 42
DEVICE = "auto"
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
def run_command(command, *, timeout=None):
    command = list(map(str, command))
    print("$", " ".join(command), flush=True)
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, timeout=timeout)
    completed.check_returncode()
    return completed


## Install and environment check

In [ ]:
run_command([sys.executable, "-m", "pip", "install", "-r", "requirements-vision-runpod.txt"], timeout=3600)
usage = shutil.disk_usage(PROJECT_ROOT)
print("free_gb:", round(usage.free / 1024**3, 2))
run_command([sys.executable, "-c", "import torch; print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"])


## Build or check downloaded-video manifest

In [ ]:
if not SAMPLE_MANIFEST.exists():
    raise FileNotFoundError(f"sample manifest not found: {SAMPLE_MANIFEST}")

if not DOWNLOAD_MANIFEST.exists():
    run_command([
        sys.executable,
        "etl/vision/download_sampled_media.py",
        "--input", SAMPLE_MANIFEST,
        "--output", DOWNLOAD_MANIFEST,
        "--download-dir", RAW_VIDEO_DIR,
        "--label-column", "coarse_label",
        "--per-label", str(PER_LABEL),
        "--split", "",
    ], timeout=None)

with DOWNLOAD_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    download_rows = list(csv.DictReader(f))
print("download_rows:", len(download_rows))
print("coarse_label_counts:", dict(Counter(row.get("coarse_label") for row in download_rows)))
print("download_status_counts:", dict(Counter(row.get("download_status") for row in download_rows)))


# Model 1: YOLO/ByteTrack clip candidates + VideoMAE

Build total 5-second clips around an accident candidate, then train VideoMAE on those clips.

In [ ]:
ACCIDENT_SOURCE = "yolo_track"  # use "center" if ByteTrack is too slow for all videos
YOLO_MODEL = "yolov8n.pt"
REBUILD_CLIPS = False  # True로 바꾸면 기존 clip manifest와 clip 파일을 다시 생성합니다.

if REBUILD_CLIPS or not CLIP_MANIFEST.exists():
    run_command([
        sys.executable,
        "etl/vision/build_training_clips.py",
        "--input", DOWNLOAD_MANIFEST,
        "--output", CLIP_MANIFEST,
        "--clip-dir", CLIP_DIR,
        "--label-column", "coarse_label",
        "--clip-sec", "5",
        "--short-video-sec", "5",
        "--accident-source", ACCIDENT_SOURCE,
        "--model-name", YOLO_MODEL,
        "--overwrite",
    ], timeout=None)
else:
    print("reuse existing clip manifest:", CLIP_MANIFEST)

with CLIP_MANIFEST.open("r", encoding="utf-8", newline="") as f:
    clip_rows = list(csv.DictReader(f))
clip_rows = [row for row in clip_rows if row.get("clip_status") == "ok"]
print("clip_rows_ok:", len(clip_rows))
print("clip_status_counts:", dict(Counter(row.get("clip_status") for row in clip_rows)))
print("clip_basis_counts:", dict(Counter(row.get("clip_basis") for row in clip_rows)))


## Shared VideoMAE helpers

In [ ]:
VIDEOMAE_MODEL_DIR = MODEL_DIR / "videomae_classification_clip5s"
EARLY_STOPPING_PATIENCE = 2

def build_videomae_command(experiment):
    command = [
        sys.executable,
        "ai/vision/train_videomae_classifier.py",
        "--manifest", CLIP_MANIFEST,
        "--root-dir", PROJECT_ROOT,
        "--output-dir", VIDEOMAE_MODEL_DIR,
        "--label-column", "coarse_label",
        "--frame-count", str(experiment["frame_count"]),
        "--epochs", str(experiment["epochs"]),
        "--batch-size", str(experiment["batch_size"]),
        "--learning-rate", str(experiment["learning_rate"]),
        "--weight-decay", str(experiment["weight_decay"]),
        "--early-stopping-patience", str(EARLY_STOPPING_PATIENCE),
        "--seed", str(SEED),
        "--device", DEVICE,
        "--num-workers", "0",
        "--no-show-progress",
    ]
    if experiment["freeze_backbone"]:
        command.append("--freeze-backbone")
    return command

def latest_run_dir(output_dir):
    runs = [path for path in output_dir.iterdir() if path.is_dir()]
    if not runs:
        raise FileNotFoundError(f"No run directories found: {output_dir}")
    return max(runs, key=lambda path: path.stat().st_mtime)

def show_run_result(run_dir):
    print("run_dir:", run_dir)
    for name in ["run_config.json", "training_history.csv"]:
        path = run_dir / name
        print("##", name, path.exists())
        if path.suffix == ".json" and path.exists():
            data = json.loads(path.read_text(encoding="utf-8"))
            for key in ["run_id", "freeze_backbone", "epochs", "batch_size", "learning_rate", "weight_decay", "best_epoch", "best_val_accuracy", "train_rows", "val_rows", "test_rows"]:
                if key in data:
                    print(key, data[key])
        elif path.exists():
            rows = list(csv.DictReader(path.open("r", encoding="utf-8")))
            for row in rows:
                print(row)
            if rows:
                print("best_val:", max(rows, key=lambda row: float(row.get("val_accuracy") or 0)))
                print("best_test:", max(rows, key=lambda row: float(row.get("test_accuracy") or 0)))


## Combination 1 - freeze baseline - define

In [ ]:
EXPERIMENT = {'name': 'videomae_clip5s_baseline_freeze_lr1e-3_e5', 'epochs': 5, 'batch_size': 2, 'learning_rate': 0.001, 'weight_decay': 0.0, 'frame_count': 16, 'freeze_backbone': True}
print(EXPERIMENT)


## Combination 1 - freeze baseline - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 1 - freeze baseline - result

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 2 - unfreeze lr 1e-4 - define

In [ ]:
EXPERIMENT = {'name': 'videomae_clip5s_exp2_lr1e-4_e10', 'epochs': 10, 'batch_size': 2, 'learning_rate': 0.0001, 'weight_decay': 0.0, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 2 - unfreeze lr 1e-4 - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 2 - unfreeze lr 1e-4 - result

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 3 - regularized lr 5e-5 - define

In [ ]:
EXPERIMENT = {'name': 'videomae_clip5s_exp3_lr5e-5_wd5e-2_e30', 'epochs': 30, 'batch_size': 2, 'learning_rate': 5e-05, 'weight_decay': 0.05, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 3 - regularized lr 5e-5 - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 3 - regularized lr 5e-5 - result

In [ ]:
show_run_result(LAST_RUN_DIR)


## Output review and next experiments

Current best VideoMAE result is Combination 3: `lr=5e-5`, `weight_decay=0.05`, unfreeze, best validation accuracy about `0.552` and test accuracy about `0.604` at epoch 5. After that, train accuracy rises while validation/test do not, so the next experiments should reduce overfitting instead of increasing epochs aggressively.

Also check the 5-second clip manifest first, because clip generation currently reduces `???` much more than the other labels. If the manifest is imbalanced, model tuning alone will not fix the result.


In [ ]:
from collections import Counter

for manifest_path in [DOWNLOAD_MANIFEST, CLIP_MANIFEST]:
    print(chr(10) + "##", manifest_path.name)
    with manifest_path.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    print("rows:", len(rows))
    for column in ["coarse_label", "split", "clip_status", "file_exists"]:
        if rows and column in rows[0]:
            print(column, dict(Counter(row.get(column) for row in rows)))


## Combination 4 - lower lr with same regularization

Keep the best direction from Combination 3, but lower the learning rate to reduce overfitting after epoch 5.


In [ ]:
EXPERIMENT = {'name': 'videomae_clip5s_exp4_lr3e-5_wd5e-2_e20', 'epochs': 20, 'batch_size': 2, 'learning_rate': 3e-05, 'weight_decay': 0.05, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 4 - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 4 - result

In [ ]:
show_run_result(LAST_RUN_DIR)


## Combination 5 - stronger regularization

Use the same learning rate as the current best run, but increase weight decay. This tests whether the overfitting after epoch 5 is mainly regularization-related.


In [ ]:
EXPERIMENT = {'name': 'videomae_clip5s_exp5_lr5e-5_wd1e-1_e20', 'epochs': 20, 'batch_size': 2, 'learning_rate': 5e-05, 'weight_decay': 0.1, 'frame_count': 16, 'freeze_backbone': False}
print(EXPERIMENT)


## Combination 5 - train

In [ ]:
run_command(build_videomae_command(EXPERIMENT), timeout=None)
LAST_RUN_DIR = latest_run_dir(VIDEOMAE_MODEL_DIR)
print("LAST_RUN_DIR:", LAST_RUN_DIR)


## Combination 5 - result

In [ ]:
show_run_result(LAST_RUN_DIR)
